In [165]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [166]:
df = pd.read_csv('train.txt', sep=';', header=None, names=['Text', 'Emotion'])

In [167]:
df.head()

,Text,Emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [168]:
df.isnull().sum()

Text       0
Emotion    0
dtype: int64

### Converting the emotions into numbers

In [170]:
unique_emotions = df['Emotion'].unique()

emotion_numbers = {}
i = 0
for emo in unique_emotions:
    emotion_numbers[emo] = i
    i += 1

In [171]:
print(emotion_numbers)

{'sadness': 0, 'anger': 1, 'love': 2, 'surprise': 3, 'fear': 4, 'joy': 5}


In [172]:
df['Emotion'] = df['Emotion'].map(emotion_numbers)

In [173]:
df

,Text,Emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


### Converting all text into lowercase

In [175]:
df['Text'] = df['Text'].apply(lambda x : x.lower())

### Remove punctuations from the texts

In [177]:
import string

def remove_punc(txt):
    return txt.translate(str.maketrans('', '', string.punctuation))

In [178]:
df['Text'] = df['Text'].apply(remove_punc)

### Remove numbers from text

In [180]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['Text'] = df['Text'].apply(remove_numbers)

### Removing emojies and special characters

In [182]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['Text'] = df['Text'].apply(remove_emojis)

### Removing stopwords

In [184]:
import nltk

In [185]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [186]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\shuvo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\shuvo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [187]:
stop_words = set(stopwords.words('english'))
len(stop_words)

198

In [188]:
df.loc[1]['Text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [189]:
def remove(txt):
    # words = word_tokenize(txt)
    words = txt.split()
    cleaned = []
    for i in words:
        if not i in stop_words:
            cleaned.append(i)
            
    return ' '.join(cleaned)

In [190]:
df['Text'] = df['Text'].apply(remove)

In [191]:
df.loc[1]['Text']

'go feeling hopeless damned hopeful around someone cares awake'

In [192]:
df.head()

,Text,Emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [193]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['Text'], df['Emotion'], test_size=0.20, random_state=42)

In [194]:
X_train

676      refers course though cant help feeling somehow...
12113                im starting feel im suffering fatigue
7077     feel like probably would liked book little bit...
13005                                  really feel awkward
12123    im feeling little grumpy today lame weather te...
                               ...                        
13418    love leave reader feeling confused slightly de...
5390                                         feel delicate
860                          starting feel little stressed
15795             feel stressed tired worn shape neglected
7270         feel someone rude wrongly done something lose
Name: Text, Length: 12800, dtype: object

In [195]:
X_test

8756                             ive made week feel beaten
4660                              feel strategy worthwhile
6095                     feel worthless weak say want find
304                                        feel clever nov
8241                      im moved ive feeling kind gloomy
                               ...                        
15578    feel useful pulpit find ironic often question ...
5746             dried bladders ready day im feeling brave
6395                             feel thrilled matter days
7624     woke morning text mr c declaring walking work ...
15245                                            feel dumb
Name: Text, Length: 3200, dtype: object

In [196]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Bag of words with Naive Bayes

In [198]:
bow_vectorizer = CountVectorizer()

In [199]:
X_train_bow = bow_vectorizer.fit_transform(X_train)

X_train_bow

<12800x13361 sparse matrix of type '<class 'numpy.int64'>'
	with 116059 stored elements in Compressed Sparse Row format>

In [200]:
X_test_bow = bow_vectorizer.transform(X_test)

X_test_bow

<3200x13361 sparse matrix of type '<class 'numpy.int64'>'
	with 26936 stored elements in Compressed Sparse Row format>

In [201]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [202]:
nb_model = MultinomialNB()

In [203]:
nb_model.fit(X_train_bow, y_train)

MultinomialNB()

In [204]:
pred_bow = nb_model.predict(X_test_bow)

print(accuracy_score(y_test, pred_bow))

0.768125


# Tf-Idf with Naive Bayes

In [206]:
tfidf_vectorizer = TfidfVectorizer()

In [207]:
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

X_train_tfidf

<12800x13361 sparse matrix of type '<class 'numpy.float64'>'
	with 116059 stored elements in Compressed Sparse Row format>

In [208]:
X_test_tfidf = tfidf_vectorizer.transform(X_test)

X_test_tfidf

<3200x13361 sparse matrix of type '<class 'numpy.float64'>'
	with 26936 stored elements in Compressed Sparse Row format>

In [209]:
nb2_model = MultinomialNB()

nb2_model.fit(X_train_tfidf, y_train)

MultinomialNB()

In [210]:
pred_bow = nb2_model.predict(X_test_tfidf)

print(accuracy_score(y_test, pred_bow))

0.6609375


#  Tf-Idf with Logistic Regression

In [253]:
from sklearn.linear_model import LogisticRegression

In [255]:
logistic_model = LogisticRegression(max_iter=1000)

In [257]:
logistic_model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [259]:
log_pred = logistic_model.predict(X_test_tfidf)

In [261]:
print(accuracy_score(y_test, log_pred))

0.8628125
